
# 練習問題: ベクトル型でfma (積和) を書く

## 目標

ベクトル型 (`__attribute__((vector_size(N)))`) の値に対して, 普通の `double` と同じ `*` `+` の記法で演算を書くと, ほぼ確実にSIMD命令 (`vfmadd...pd` など) に変換されることを体験する.

## 課題

### C++ (ベクトル型)

`vector_fma.cpp` には, `double` 8つ分 (512 bit) のベクトル型 `doublev` を引数にとり, 積和 (fma, fused multiply-add) `a * b + c` を計算する関数 `vector_fma` がある.
本体が `TODO` で空になっている.

コメント `TODO` の指示に従って, **`a * b + c` を返す1行を書け**.

```cpp
return a * b + c;
```

a, b, c はいずれもベクトル型なので, `*` `+` は要素ごとのSIMD演算になる. それ以外のコードを変更する必要はない.

### Fortran (`!$omp simd`)

<font color="red">重要:</font> `__attribute__((vector_size(N)))` によるベクトル型は **C/C++独自の拡張** であり, Fortranには相当する機能が無い.
そこでFortran版 `vector_fma.f90` では, 同じ `y(i) = a(i) * b(i) + c(i)` の計算をループで書き, `!$omp simd` でSIMD化する.
`TODO` の指示に従って doループの直前に **`!$omp simd` を1行追加** せよ.

## コンパイルと実行

```
# C++ (ベクトル型)
nvc++ -fast -Mkeepasm -Minfo -Mneginfo -c vector_fma.cpp

# Fortran (!$omp simd を解釈させるため -mp=multicore が必要)
nvfortran -fast -mp=multicore -Mkeepasm -Minfo -Mneginfo -c vector_fma.f90
```

生成されたアセンブリを確認する.

```
cat vector_fma.s
```

## 期待される結果

C++版では `vector_fma.s` の中に, 512 bit レジスタ `%zmm` を使った積和命令

```
vfmadd213pd %zmm2, %zmm1, %zmm0   # zmm0 = (zmm1 * zmm0) + zmm2
```

が現れる. `pd` は _packed double precision_ の略である.
自動ベクトル化では256 bit命令 (`%ymm`) しか使われない例があったが, ベクトル型を明示的に使うことで望み通りの512 bit命令を確実に引き出せていることを確認せよ.

Fortran版では, `-Minfo` の出力に "Generated vector ..." のようなメッセージが現れ, `vector_fma.s` に `pd` 付きのSIMD命令が生成されていればSIMD化に成功している.



# 1. このノートについて
- この問題は「生成されたアセンブリ(機械語)を見て, SIMD命令に変換されたかを確認する」ことが目的。
- 関数だけのソース(`main` 無し)を `-c` でオブジェクトコンパイルし, `-Mkeepasm` で残る `.s` を読む。
- 実行はしない(`-c` なので実行ファイルは作らない)。

# 2. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)


In [ ]:
import heytutor
import wisteria_submit

# 3. C++ ベースコード

In [ ]:
%%writefile_ vector_fma.cpp
#include <cstdio>

/* double 8 つ分 (8 × 8 = 64 バイト = 512 bit) のベクトル型 */
typedef double doublev __attribute__((vector_size(64)));

/* a, b, c はいずれも double 8 つ分のベクトル型.
   a*b+c を要素ごとの演算 (fma) としてベクトルのまま計算して返す. */
doublev vector_fma(doublev a, doublev b, doublev c) {
  // TODO: a * b + c を返す1行を書け.
}

## 3-1. コンパイル (アセンブリを残す)
- `-c` : 実行ファイルを作らずオブジェクトファイルまで (`main` が無くてよい)
- `-Mkeepasm` : 生成アセンブリ `vector_fma.s` を残す
- `-Minfo` / `-Mneginfo` : ベクトル化に成功/失敗したことを報告する

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore -Mkeepasm -Minfo -Mneginfo -c vector_fma.cpp

## 3-2. 生成されたアセンブリを見る
- `pd` の付いた packed 命令 (`vmulpd`, `vaddpd`, `vfmadd...pd`) や, `%ymm`/`%zmm` レジスタが出ていればSIMD化されている。

In [ ]:
%%bash_
cat vector_fma.s

# 4. Fortran ベースコード

In [ ]:
%%writefile_ vector_fma.f90
! Fortran には C/C++ のベクトル型拡張 (vector_size) に相当する機能が無い.
! そこで, 同じ y(i) = a(i) * b(i) + c(i) の積和 (fma) をループで書き,
! !$omp simd で SIMD 化する.
subroutine vector_fma(n, a, b, c, y)
  implicit none
  integer(8), intent(in) :: n
  real(8), intent(in) :: a(n), b(n), c(n)
  real(8), intent(out) :: y(n)
  integer(8) :: i
  ! TODO: 下の do ループの直前に !$omp simd を1行追加し, このループをSIMD化せよ.
  do i = 1, n
     y(i) = a(i) * b(i) + c(i)
  end do
end subroutine vector_fma

## 4-1. コンパイル (アセンブリを残す)

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore -Mkeepasm -Minfo -Mneginfo -c vector_fma.f90

## 4-2. 生成されたアセンブリを見る

In [ ]:
%%bash_
cat vector_fma.s


# 5. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:vector_fma.cpp}`, `{file:vector_fma.s}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 5-1. 一般的な質問

In [ ]:
%%hey

SIMD命令(packed double, vfmadd...pd など)って何?

## 5-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 5-3. 困ったときのヘルプ
- コンパイル時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:vector_fma.cpp}

コマンドと実行結果:
{bash[-1]}

## 5-4. アセンブリについて質問

In [ ]:
%%hey

生成されたアセンブリを説明して。SIMD化されている?

ソース:
{file:vector_fma.cpp}

アセンブリ:
{file:vector_fma.s}